In [1]:
import os
from unittest import result
from urllib.request import localhost

from rich.jupyter import display
from sqlalchemy import create_engine, text

In [2]:
from sqlalchemy import URL, create_engine, text
import os
url = URL.create(
    drivername="postgresql+psycopg",
    username=os.environ['POSTGRES_USER'],
    password=os.environ['POSTGRES_PASSWORD'],
    host='localhost',
    database="ecommerce_analytics",
    port=5432
)
engine = create_engine(url)


In [3]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT current_database();"))
    print("Connected to:", result.scalar())

Connected to: ecommerce_analytics


In [4]:
import pandas as pd

df = pd.read_csv(
    r"cleaned_online_retail.csv",
    encoding="ISO-8859-1"
)

print(df.shape)
df.head()


(530104, 10)


C:\Users\Arsh\AppData\Local\Temp\ipykernel_10212\3086200135.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Month
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 530104 entries, 0 to 530103
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    530104 non-null  object 
 1   StockCode    530104 non-null  object 
 2   Description  530104 non-null  object 
 3   Quantity     530104 non-null  int64  
 4   InvoiceDate  530104 non-null  object 
 5   UnitPrice    530104 non-null  float64
 6   CustomerID   397884 non-null  float64
 7   Country      530104 non-null  object 
 8   Revenue      530104 non-null  float64
 9   Month        530104 non-null  object 
dtypes: float64(3), int64(1), object(6)
memory usage: 40.4+ MB


In [6]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = df["CustomerID"].astype("Int64")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 530104 entries, 0 to 530103
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    530104 non-null  object        
 1   StockCode    530104 non-null  object        
 2   Description  530104 non-null  object        
 3   Quantity     530104 non-null  int64         
 4   InvoiceDate  530104 non-null  datetime64[ns]
 5   UnitPrice    530104 non-null  float64       
 6   CustomerID   397884 non-null  Int64         
 7   Country      530104 non-null  object        
 8   Revenue      530104 non-null  float64       
 9   Month        530104 non-null  object        
dtypes: Int64(1), datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 40.9+ MB


In [7]:

try:
    df.to_sql(
        name="online_retail",
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=5000
    )
    print("Data imported successfully!")

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error message:", str(e)[:1500])


Data imported successfully!


In [8]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) from online_retail;"))
    print("Total rows:", result.scalar())

Total rows: 530104


In [9]:
with engine.connect() as connection:
    result = connection.execute(text('SELECT SUM("Revenue") from online_retail;'))
    print(result.scalar())

10666684.544000275


In [10]:

query = text("""
    SELECT
        "Country",
        ROUND(SUM("Revenue")::numeric, 2) AS total_revenue
    FROM online_retail
    GROUP BY "Country"
    ORDER BY total_revenue DESC
    LIMIT 5;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('United Kingdom', Decimal('9025222.08'))
('Netherlands', Decimal('285446.34'))
('EIRE', Decimal('283453.96'))
('Germany', Decimal('228867.14'))
('France', Decimal('209715.11'))


In [11]:

query = text("""
    WITH monthly_sales AS (
        SELECT
            "Month",
            SUM("Revenue") AS monthly_revenue
        FROM online_retail
        GROUP BY "Month"
    ),
    monthly_comparison AS (
        SELECT
            "Month",
            monthly_revenue,
            LAG(monthly_revenue) OVER (
                ORDER BY "Month"
            ) AS previous_revenue
        FROM monthly_sales
    )
    SELECT
        "Month",
        ROUND(monthly_revenue::numeric, 2) AS revenue,
        ROUND(previous_revenue::numeric, 2) AS previous_revenue,
        ROUND(
            (monthly_revenue - previous_revenue)::numeric, 2
        ) AS revenue_change,
        ROUND(
            (
                (monthly_revenue - previous_revenue)
                / NULLIF(previous_revenue, 0) * 100
            )::numeric, 2
        ) AS growth_percentage
    FROM monthly_comparison
    ORDER BY "Month" ASC;
""")

with engine.connect() as connection:
    result = connection.execute(query)

    for row in result:
        print(row)


('2010-12', Decimal('823746.14'), None, None, None)
('2011-01', Decimal('691364.56'), Decimal('823746.14'), Decimal('-132381.58'), Decimal('-16.07'))
('2011-02', Decimal('523631.89'), Decimal('691364.56'), Decimal('-167732.67'), Decimal('-24.26'))
('2011-03', Decimal('717639.36'), Decimal('523631.89'), Decimal('194007.47'), Decimal('37.05'))
('2011-04', Decimal('537808.62'), Decimal('717639.36'), Decimal('-179830.74'), Decimal('-25.06'))
('2011-05', Decimal('770536.02'), Decimal('537808.62'), Decimal('232727.40'), Decimal('43.27'))
('2011-06', Decimal('761739.90'), Decimal('770536.02'), Decimal('-8796.12'), Decimal('-1.14'))
('2011-07', Decimal('719221.19'), Decimal('761739.90'), Decimal('-42518.71'), Decimal('-5.58'))
('2011-08', Decimal('759138.38'), Decimal('719221.19'), Decimal('39917.19'), Decimal('5.55'))
('2011-09', Decimal('1058590.17'), Decimal('759138.38'), Decimal('299451.79'), Decimal('39.45'))
('2011-10', Decimal('1154979.30'), Decimal('1058590.17'), Decimal('96389.13'), D

In [12]:

query = text("""
    WITH monthly_sales AS (
        SELECT
            "Month",
            SUM("Revenue") AS monthly_revenue
        FROM online_retail
        GROUP BY "Month"
    ),
    monthly_comparison AS (
        SELECT
            "Month",
            monthly_revenue,
            LAG(monthly_revenue) OVER (
                ORDER BY "Month"
            ) AS previous_revenue
        FROM monthly_sales
    )
    SELECT
        "Month",
        ROUND(
            (monthly_revenue - previous_revenue)::numeric, 2
        ) AS revenue_change
    FROM monthly_comparison
    WHERE previous_revenue IS NOT NULL
    ORDER BY revenue_change ASC
    LIMIT 1;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)



('2011-12', Decimal('-870703.65'))


In [13]:

query = text("""
    SELECT
        "Month",
        COUNT(*) AS total_rows,
        SUM("Revenue") AS monthly_revenue
    FROM online_retail
    WHERE "Month" IN ('2011-11', '2011-12')
    GROUP BY "Month"
    ORDER BY "Month";
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('2011-11', 83369, 1509496.3300000415)
('2011-12', 25111, 638792.6800000065)


In [14]:

query = text("""
    SELECT
        MIN("InvoiceDate") AS first_transaction,
        MAX("InvoiceDate") AS last_transaction
    FROM online_retail;
""")

with engine.connect() as connection:
    print(connection.execute(query).fetchone())


(datetime.datetime(2010, 12, 1, 8, 26), datetime.datetime(2011, 12, 9, 12, 50))


In [15]:
# we see dataset ends on 9 december

In [16]:

query = text("""
    SELECT
        "Month",
        SUM("Revenue") AS monthly_revenue,
        COUNT(DISTINCT CAST("InvoiceDate" AS DATE)) AS active_days,
        ROUND(
            (
                SUM("Revenue") /
                NULLIF(
                    COUNT(DISTINCT CAST("InvoiceDate" AS DATE)),
                    0
                )
            )::numeric,
            2
        ) AS revenue_per_active_day
    FROM online_retail
    WHERE "Month" IN ('2011-11', '2011-12')
    GROUP BY "Month"
    ORDER BY "Month";
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('2011-11', 1509496.3299997416, 26, Decimal('58057.55'))
('2011-12', 638792.6800000162, 8, Decimal('79849.09'))


In [17]:
# average revenue per active day actually increased thats crazy

In [18]:

query = text("""
    SELECT
        CAST("InvoiceDate" AS DATE) AS sales_date,
        SUM("Revenue") AS daily_revenue,
        COUNT(*) AS transaction_rows
    FROM online_retail
    GROUP BY CAST("InvoiceDate" AS DATE)
    ORDER BY daily_revenue DESC
    LIMIT 5;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


(datetime.date(2011, 12, 9), 200920.60000000027, 1625)
(datetime.date(2011, 11, 14), 114419.89000000004, 3521)
(datetime.date(2011, 9, 20), 109612.03, 1708)
(datetime.date(2010, 12, 7), 99618.20000000104, 2862)
(datetime.date(2011, 1, 18), 95978.05000000022, 1428)


In [19]:
#lets investigate the highest revenue day

query = text("""
    SELECT
        "Country",
        ROUND(SUM("Revenue")::numeric, 2) AS daily_revenue,
        COUNT(*) AS transaction_rows
    FROM online_retail
    WHERE CAST("InvoiceDate" AS DATE) = '2011-12-09'
    GROUP BY "Country"
    ORDER BY daily_revenue DESC
    LIMIT 5;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('United Kingdom', Decimal('196134.10'), 1445)
('Norway', Decimal('2638.69'), 66)
('Germany', Decimal('1689.72'), 82)
('France', Decimal('249.45'), 15)
('Belgium', Decimal('208.64'), 17)


In [20]:
#most of it came from UK
#lets see top products

query = text("""
    SELECT
        "StockCode",
        "Description",
        SUM("Revenue") AS product_revenue,
        SUM("Quantity") AS units_sold
    FROM online_retail
    WHERE CAST("InvoiceDate" AS DATE) = '2011-12-09'
      AND "Country" = 'United Kingdom'
    GROUP BY "StockCode", "Description"
    ORDER BY product_revenue DESC
    LIMIT 5;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('23843', 'PAPER CRAFT , LITTLE BIRDIE', 168469.6, Decimal('80995'))
('DOT', 'DOTCOM POSTAGE', 2647.34, Decimal('2'))
('21137', 'BLACK RECORD COVER FRAME', 669.57, Decimal('189'))
('22086', "PAPER CHAIN KIT 50'S CHRISTMAS ", 591.9, Decimal('114'))
('20749', 'ASSORTED COLOUR MINI CASES', 533.4, Decimal('84'))


In [21]:
#168k was this little birdie thing

query = text("""
    SELECT
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "Revenue",
        "InvoiceDate"
    FROM online_retail
    WHERE CAST("InvoiceDate" AS DATE) = '2011-12-09'
      AND "Country" = 'United Kingdom'
      AND "StockCode" = '23843'
    ORDER BY "Quantity" DESC
    LIMIT 10;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('581483', '23843', 'PAPER CRAFT , LITTLE BIRDIE', 80995, 2.08, 168469.6, datetime.datetime(2011, 12, 9, 9, 15))


In [22]:
#all of it was ordered once
#pretty sussy
#check if its an outlier

query = text("""
    SELECT
        COUNT(*) AS total_records,
        MIN("Quantity") AS minimum_quantity,
        ROUND(AVG("Quantity")::numeric, 2) AS average_quantity,
        MAX("Quantity") AS maximum_quantity,
        PERCENTILE_CONT(0.5)
            WITHIN GROUP (ORDER BY "Quantity") AS median_quantity
    FROM online_retail
    WHERE "StockCode" = '23843'
      AND "Quantity" > 0;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


(1, 80995, Decimal('80995.00'), 80995, 80995.0)


In [23]:
# we see theres only one record

query = text("""
    SELECT
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "Revenue",
        "Country"
    FROM online_retail
    WHERE "InvoiceNo" = '581483'
    ORDER BY "Revenue" DESC;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('581483', '23843', 'PAPER CRAFT , LITTLE BIRDIE', 80995, 2.08, 168469.6, 'United Kingdom')


In [24]:
#we check if the invoice contained other products
#finding outliers quantity wise

query = text("""
    WITH quantity_threshold AS (
        SELECT
            PERCENTILE_CONT(0.99)
                WITHIN GROUP (ORDER BY "Quantity") AS p99
        FROM online_retail
        WHERE "Quantity" > 0
    )
    SELECT
        r."InvoiceNo",
        r."StockCode",
        r."Description",
        r."Quantity",
        r."UnitPrice",
        r."Revenue",
        q.p99 AS quantity_99th_percentile
    FROM online_retail AS r
    CROSS JOIN quantity_threshold AS q
    WHERE r."Quantity" > q.p99
    ORDER BY r."Quantity" DESC
    LIMIT 10;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('581483', '23843', 'PAPER CRAFT , LITTLE BIRDIE', 80995, 2.08, 168469.6, 100.0)
('541431', '23166', 'MEDIUM CERAMIC TOP STORAGE JAR', 74215, 1.04, 77183.6, 100.0)
('573008', '84077', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 4800, 0.21, 1008.0, 100.0)
('554868', '22197', 'SMALL POPCORN HOLDER', 4300, 0.72, 3096.0, 100.0)
('544612', '22053', 'EMPIRE DESIGN ROSETTE', 3906, 0.82, 3202.92, 100.0)
('560599', '18007', 'ESSENTIAL BALM 3.5g TIN IN ENVELOPE', 3186, 0.06, 191.16, 100.0)
('540815', '21108', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, 2.1, 6539.400000000001, 100.0)
('550461', '21108', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, 2.1, 6539.400000000001, 100.0)
('573995', '16014', 'SMALL CHINESE STYLE SCISSOR', 3000, 0.32, 960.0, 100.0)
('562439', '84879', 'ASSORTED COLOUR BIRD ORNAMENT', 2880, 1.45, 4176.0, 100.0)


In [25]:
#zscore

query = text("""
    WITH quantity_stats AS (
        SELECT
            AVG("Quantity") AS avg_quantity,
            STDDEV("Quantity") AS std_quantity
        FROM online_retail
        WHERE "Quantity" > 0
    )
    SELECT
        r."InvoiceNo",
        r."Description",
        r."Quantity",
        ROUND(s.avg_quantity::numeric, 2) AS avg_quantity,
        ROUND(s.std_quantity::numeric, 2) AS std_quantity,
        ROUND(
            (
                (r."Quantity" - s.avg_quantity)
                / NULLIF(s.std_quantity, 0)
            )::numeric,
            2
        ) AS z_score
    FROM online_retail AS r
    CROSS JOIN quantity_stats AS s
    WHERE r."Quantity" > 0
    ORDER BY r."Quantity" DESC
    LIMIT 10;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('581483', 'PAPER CRAFT , LITTLE BIRDIE', 80995, Decimal('10.54'), Decimal('155.52'), Decimal('520.72'))
('541431', 'MEDIUM CERAMIC TOP STORAGE JAR', 74215, Decimal('10.54'), Decimal('155.52'), Decimal('477.13'))
('573008', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 4800, Decimal('10.54'), Decimal('155.52'), Decimal('30.80'))
('554868', 'SMALL POPCORN HOLDER', 4300, Decimal('10.54'), Decimal('155.52'), Decimal('27.58'))
('544612', 'EMPIRE DESIGN ROSETTE', 3906, Decimal('10.54'), Decimal('155.52'), Decimal('25.05'))
('560599', 'ESSENTIAL BALM 3.5g TIN IN ENVELOPE', 3186, Decimal('10.54'), Decimal('155.52'), Decimal('20.42'))
('550461', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, Decimal('10.54'), Decimal('155.52'), Decimal('19.95'))
('540815', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, Decimal('10.54'), Decimal('155.52'), Decimal('19.95'))
('573995', 'SMALL CHINESE STYLE SCISSOR', 3000, Decimal('10.54'), Decimal('155.52'), Decimal('19.22'))
('562439', 'ASSORTED COLOUR BIRD ORNAMENT', 2880, D

In [26]:

query = text("""
    SELECT
        "InvoiceNo",
        "Description",
        "Quantity",
        "Revenue",
        CASE
            WHEN "Quantity" > 100 THEN 'Review: High Quantity'
            ELSE 'Normal'
        END AS anomaly_status
    FROM online_retail
    WHERE "Quantity" > 100
    ORDER BY "Quantity" DESC
    LIMIT 20;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('581483', 'PAPER CRAFT , LITTLE BIRDIE', 80995, 168469.6, 'Review: High Quantity')
('541431', 'MEDIUM CERAMIC TOP STORAGE JAR', 74215, 77183.6, 'Review: High Quantity')
('573008', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 4800, 1008.0, 'Review: High Quantity')
('554868', 'SMALL POPCORN HOLDER', 4300, 3096.0, 'Review: High Quantity')
('544612', 'EMPIRE DESIGN ROSETTE', 3906, 3202.92, 'Review: High Quantity')
('560599', 'ESSENTIAL BALM 3.5g TIN IN ENVELOPE', 3186, 191.16, 'Review: High Quantity')
('550461', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, 6539.400000000001, 'Review: High Quantity')
('540815', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, 6539.400000000001, 'Review: High Quantity')
('573995', 'SMALL CHINESE STYLE SCISSOR', 3000, 960.0, 'Review: High Quantity')
('562439', 'ASSORTED COLOUR BIRD ORNAMENT', 2880, 4176.0, 'Review: High Quantity')
('536830', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 2880, 518.4, 'Review: High Quantity')
('554272', 'PACK OF 60 PINK PAISLEY CAKE CASES', 2700, 11

In [27]:

query = text("""
    SELECT
        "InvoiceNo",
        "Description",
        "Quantity",
        "Revenue",
        CASE
            WHEN "Quantity" > 1000 THEN 'High Risk'
            WHEN "Quantity" > 100 THEN 'Review'
            ELSE 'Normal'
        END AS anomaly_status
    FROM online_retail
    WHERE "Quantity" > 100
    ORDER BY "Quantity" DESC
    LIMIT 20;
""")

with engine.connect() as connection:
    for row in connection.execute(query):
        print(row)


('581483', 'PAPER CRAFT , LITTLE BIRDIE', 80995, 168469.6, 'High Risk')
('541431', 'MEDIUM CERAMIC TOP STORAGE JAR', 74215, 77183.6, 'High Risk')
('573008', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 4800, 1008.0, 'High Risk')
('554868', 'SMALL POPCORN HOLDER', 4300, 3096.0, 'High Risk')
('544612', 'EMPIRE DESIGN ROSETTE', 3906, 3202.92, 'High Risk')
('560599', 'ESSENTIAL BALM 3.5g TIN IN ENVELOPE', 3186, 191.16, 'High Risk')
('550461', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, 6539.400000000001, 'High Risk')
('540815', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, 6539.400000000001, 'High Risk')
('573995', 'SMALL CHINESE STYLE SCISSOR', 3000, 960.0, 'High Risk')
('562439', 'ASSORTED COLOUR BIRD ORNAMENT', 2880, 4176.0, 'High Risk')
('536830', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 2880, 518.4, 'High Risk')
('554272', 'PACK OF 60 PINK PAISLEY CAKE CASES', 2700, 1134.0, 'High Risk')
('543057', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 2592, 544.3199999999999, 'High Risk')
('539101', 'GROW A FLYTRAP 

In [32]:
def detect_anamolies(limit=10):
    query = text("""
        SELECT
        "InvoiceNo",
        "Description",
        "Quantity",
        "Revenue",
        CASE
            WHEN "Quantity" > 1000 THEN 'High Risk'
            WHEN "Quantity" > 100 THEN 'Review'
            ELSE 'Normal'
        END AS anomaly_status
    FROM online_retail
    WHERE "Quantity" > 100
    ORDER BY "Quantity" DESC
    LIMIT :limit;
    """)
    with engine.connect() as connection:
        results = connection.execute(
            query, {"limit": limit}
        ).fetchall()

    return results

In [34]:
anamoly = detect_anamolies(20)
for row in anamoly:
    print(row)

('581483', 'PAPER CRAFT , LITTLE BIRDIE', 80995, 168469.6, 'High Risk')
('541431', 'MEDIUM CERAMIC TOP STORAGE JAR', 74215, 77183.6, 'High Risk')
('573008', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 4800, 1008.0, 'High Risk')
('554868', 'SMALL POPCORN HOLDER', 4300, 3096.0, 'High Risk')
('544612', 'EMPIRE DESIGN ROSETTE', 3906, 3202.92, 'High Risk')
('560599', 'ESSENTIAL BALM 3.5g TIN IN ENVELOPE', 3186, 191.16, 'High Risk')
('550461', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, 6539.400000000001, 'High Risk')
('540815', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, 6539.400000000001, 'High Risk')
('573995', 'SMALL CHINESE STYLE SCISSOR', 3000, 960.0, 'High Risk')
('562439', 'ASSORTED COLOUR BIRD ORNAMENT', 2880, 4176.0, 'High Risk')
('536830', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 2880, 518.4, 'High Risk')
('554272', 'PACK OF 60 PINK PAISLEY CAKE CASES', 2700, 1134.0, 'High Risk')
('543057', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 2592, 544.3199999999999, 'High Risk')
('539101', 'GROW A FLYTRAP 

In [37]:

def detect_anamolies(threshold=100, limit=10):
    query = text("""
        SELECT
        "InvoiceNo",
        "Description",
        "Quantity",
        "Revenue",
        CASE
            WHEN "Quantity" > :high_threshold THEN 'High Risk'
            WHEN "Quantity" > :threshold THEN 'Review'
            ELSE 'Normal'
        END AS anomaly_status
    FROM online_retail
    WHERE "Quantity" > 100
    ORDER BY "Quantity" DESC
    LIMIT :limit;
    """)
    with engine.connect() as connection:
        results = connection.execute(
            query, {"limit": limit,
                    "threshold": threshold,
                    "high_threshold": threshold*10}
        ).fetchall()

    return results

anamoly = detect_anomalies(500, 20)
for row in anamoly:
    print(row)

('581483', 'PAPER CRAFT , LITTLE BIRDIE', 80995, 168469.6, 'High Risk')
('541431', 'MEDIUM CERAMIC TOP STORAGE JAR', 74215, 77183.6, 'High Risk')
('573008', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 4800, 1008.0, 'Review')
('554868', 'SMALL POPCORN HOLDER', 4300, 3096.0, 'Review')
('544612', 'EMPIRE DESIGN ROSETTE', 3906, 3202.92, 'Review')
('560599', 'ESSENTIAL BALM 3.5g TIN IN ENVELOPE', 3186, 191.16, 'Review')
('550461', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, 6539.400000000001, 'Review')
('540815', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 3114, 6539.400000000001, 'Review')
('573995', 'SMALL CHINESE STYLE SCISSOR', 3000, 960.0, 'Review')
('562439', 'ASSORTED COLOUR BIRD ORNAMENT', 2880, 4176.0, 'Review')
('536830', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 2880, 518.4, 'Review')
('554272', 'PACK OF 60 PINK PAISLEY CAKE CASES', 2700, 1134.0, 'Review')
('543057', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 2592, 544.3199999999999, 'Review')
('544152', 'ESSENTIAL BALM 3.5g TIN IN ENVELOPE', 2400, 144.

In [39]:
def detect_anamolies_pd(threshold=100, limit=10):
    query = text("""
        SELECT
        "InvoiceNo",
        "Description",
        "Quantity",
        "Revenue",
        CASE
            WHEN "Quantity" > :high_threshold THEN 'High Risk'
            WHEN "Quantity" > :threshold THEN 'Review'
            ELSE 'Normal'
        END AS anomaly_status
    FROM online_retail
    WHERE "Quantity" > 100
    ORDER BY "Quantity" DESC
    LIMIT :limit;
    """)
    with engine.connect() as connection:
        df_results = pd.read_sql(
            query,
            connection,
            params={
                "threshold": threshold,
                "high_threshold": threshold * 10,
                "limit": limit
            }
        )
        return df_results
anamoly = detect_anamolies_pd(500, 20)
display(anamoly)

,InvoiceNo,Description,Quantity,Revenue,anomaly_status
0,581483,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,High Risk
1,541431,MEDIUM CERAMIC TOP STORAGE JAR,74215,77183.60,High Risk
2,573008,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,1008.00,Review
3,554868,SMALL POPCORN HOLDER,4300,3096.00,Review
4,544612,EMPIRE DESIGN ROSETTE,3906,3202.92,Review
5,560599,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,191.16,Review
6,550461,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,6539.40,Review
7,540815,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,6539.40,Review
8,573995,SMALL CHINESE STYLE SCISSOR,3000,960.00,Review
9,562439,ASSORTED COLOUR BIRD ORNAMENT,2880,4176.00,Review


In [40]:

def get_monthly_revenue():
    query = text("""
        WITH monthly_sales AS (
            SELECT
                "Month",
                SUM("Revenue") AS monthly_revenue
            FROM online_retail
            GROUP BY "Month"
        ),
        monthly_comparison AS (
            SELECT
                "Month",
                monthly_revenue,
                LAG(monthly_revenue) OVER (
                    ORDER BY "Month"
                ) AS previous_revenue
            FROM monthly_sales
        )
        SELECT
            "Month",
            ROUND(monthly_revenue::numeric, 2) AS revenue,
            ROUND(previous_revenue::numeric, 2) AS previous_revenue,
            ROUND(
                (
                    (monthly_revenue - previous_revenue)
                    / NULLIF(previous_revenue, 0) * 100
                )::numeric,
                2
            ) AS growth_percentage
        FROM monthly_comparison
        ORDER BY "Month";
    """)

    with engine.connect() as connection:
        result = pd.read_sql(query, connection)

    return result


In [41]:
monthly_df = get_monthly_revenue()
display(monthly_df)

,Month,revenue,previous_revenue,growth_percentage
0,2010-12,823746.14,NaN,NaN
1,2011-01,691364.56,823746.14,-16.07
2,2011-02,523631.89,691364.56,-24.26
3,2011-03,717639.36,523631.89,37.05
4,2011-04,537808.62,717639.36,-25.06
5,2011-05,770536.02,537808.62,43.27
6,2011-06,761739.90,770536.02,-1.14
7,2011-07,719221.19,761739.90,-5.58
8,2011-08,759138.38,719221.19,5.55
9,2011-09,1058590.17,759138.38,39.45
